In [8]:
#%pip install python-dotenv
#%pip install kagglehub
#%pip install pretty_midi tqdm
%pip install kaggle


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [18]:
import os
from dotenv import load_dotenv
import pretty_midi
import pandas as pd
import glob
import numpy as np
from collections import Counter


In [ ]:
# 1. Load the environment variables BEFORE importing kaggle
load_dotenv()

# Verify they loaded successfully (optional, but good for debugging)
# print("Username loaded:", os.environ.get('KAGGLE_USERNAME') is not None)

# 2. Now it's safe to import Kaggle. 
# It will automatically detect KAGGLE_USERNAME and KAGGLE_KEY from the environment.
import kaggle

# 3. Authenticate and test
kaggle.api.authenticate()
print("Authenticated to Kaggle successfully!")



Authenticated to Kaggle successfully!


In [20]:
kaggle.api.dataset_download_files(
    'godsonajodo/anime-midi', 
    path='./anime_midi_data', 
    unzip=True
)

print("Download and extraction complete!")

Dataset URL: https://www.kaggle.com/datasets/godsonajodo/anime-midi
Download and extraction complete!


In [11]:
import pretty_midi
import glob

# Find all mid files in your extracted folder
midi_files = glob.glob('./nintendo_midi_data/**/*.mid', recursive=True)

if midi_files:
    # Load the first midi file found
    midi_data = pretty_midi.PrettyMIDI(midi_files[0])
    
    print(f"Loaded File: {midi_files[0]}")
    print(f"Duration: {midi_data.get_end_time():.2f} seconds")
    print(f"Tempo (BPM): {midi_data.estimate_tempo():.0f}")
    print("\nInstruments found:")
    
    for instrument in midi_data.instruments:
        print(f" - {instrument.name} (Program {instrument.program}) | Total Notes: {len(instrument.notes)}")
else:
    print("No MIDI files found. Double check your unzipped folder path!")

Loaded File: ./nintendo_midi_data\nin_midi_files\nin_midi_files\Microsoft\XBOX\Halo2\TheLastSpartan.mid
Duration: 129.00 seconds
Tempo (BPM): 135

Instruments found:
 - SmartMusic SoftSynth (Program 0) | Total Notes: 390


In [20]:
def analyze_tracks_and_vocab(file_path):
    try:
        pm = pretty_midi.PrettyMIDI(file_path)
    except Exception:
        # Returns None if a fan-made MIDI file is corrupt
        return None
    
    duration = pm.get_end_time()
    if duration == 0:
        return None

    # Core statistical accumulators
    all_pitches = []
    all_durations = []
    start_times = []
    
    melody_tracks_count = 0
    harmony_tracks_count = 0
    
    # Analyze individual tracks (instruments)
    for instrument in pm.instruments:
        if instrument.is_drum:
            continue
            
        notes = instrument.notes
        if len(notes) == 0:
            continue
            
        track_pitches = [n.pitch for n in notes]
        track_start_times = [n.start for n in notes]
        track_durations = [round(n.end - n.start, 3) for n in notes]
        
        all_pitches.extend(track_pitches)
        all_durations.extend(track_durations)
        start_times.extend(track_start_times)
        
        # Heuristic for Task 2 Track Separation:
        # Calculate track polyphony (average notes playing at the exact same start time)
        time_counts = Counter(track_start_times)
        avg_track_polyphony = np.mean(list(time_counts.values()))
        
        # If notes rarely overlap, it's a monophonic melody line
        if avg_track_polyphony < 1.1:
            melody_tracks_count += 1
        else:
            harmony_tracks_count += 1

    if not all_pitches:
        return None

    # Calculate overall polyphony across the entire song
    overall_time_counts = Counter(start_times)
    avg_overall_polyphony = np.mean(list(overall_time_counts.values()))

    return {
        "file_name": file_path.split("/")[-1],
        "duration": duration,
        "total_notes": len(all_pitches),
        "unique_pitches_in_file": len(set(all_pitches)),
        "min_pitch": min(all_pitches),
        "max_pitch": max(all_pitches),
        "unique_durations_in_file": len(set(all_durations)),
        "avg_polyphony": avg_overall_polyphony,
        "melody_tracks": melody_tracks_count,
        "harmony_tracks": harmony_tracks_count,
        "all_pitches": all_pitches,
        "all_durations": all_durations
    }

# ---------------------------------------------------------
# Execution & Table Generation
# ---------------------------------------------------------

# Replace with your actual list of paths if already loaded
midi_files = glob.glob('./nintendo_midi_data/**/*.mid', recursive=True)

# Run pipeline over your dataset (using a sample of 100 for speed; remove [:100] for full dataset)
processed_data = []
corrupt_count = 0

for f in midi_files[:100]:
    metrics = analyze_tracks_and_vocab(f)
    if metrics is not None:
        processed_data.append(metrics)
    else:
        corrupt_count += 1

df = pd.DataFrame(processed_data)

# Flatten master lists to calculate global vocabularies
global_pitches = [pitch for song in df['all_pitches'] for pitch in song]
global_durations = [dur for song in df['all_durations'] for dur in song]

print("--- PIPELINE PROCESSING COMPLETE ---\n")

--- PIPELINE PROCESSING COMPLETE ---



In [17]:
# Assuming you built a list of dictionaries from your parsed MIDI files
df = pd.DataFrame(df_eda)

# Example: Generating the Vocabulary & Sequence Table
summary_table = pd.DataFrame({
    'Metric': [
        'Total Usable Sequences',
        'Max Sequence Length (Notes)',
        'Avg Sequence Length',
        'Unique Pitch Tokens (Vocab Size)',
        'Most Common MIDI Pitch',
        'Overall Pitch Range'
    ],
    'Value': [
        len(df),
        df['total_notes'].max(),
        round(df['total_notes'].mean(), 2),
        df['unique_pitches'].nunique(), # Maps to output layer size
        df['dominant_pitch'].mode()[0],
        f"{df['min_pitch'].min()} - {df['max_pitch'].max()}"
    ]
})

print(summary_table.to_string(index=False))

KeyError: 'unique_pitches'